# TERRA tutorial - in-silico perturbation (test section)

Knock out glomerular filtration-barrier genes (**NPNT, WT1, MAGI1**) *in silico* across the section and measure the effect on each niche's representation with a **W2 (Sinkhorn) distance** between unperturbed and perturbed embeddings.

> **Requirements:** an NVIDIA GPU and TERRA installed **with the perturbation extra** see the [installation guide](https://terra-st.readthedocs.io/en/latest/installation.html). This tutorial also uses `gdown` to download the demo data (`pip install gdown`). This notebook needs **geomloss** for the perturbation distances  it is installed automatically by `terra-st[perturb]`; if you installed plain `terra-st`, add it with `pip install geomloss`.

This tutorial runs **end-to-end on a de-identified test kidney section we provide** using the public [`lotfollahi-lab/TERRA-96M`](https://huggingface.co/lotfollahi-lab/TERRA-96M) model. Expensive steps (tokenisation, perturbation) are **cached** locally the first run computes them, later runs (and the other tutorials) reuse the cache.

## 1. Setup

In [ ]:
# Colab / fresh environment only. Skip if you already installed TERRA per the installation guide.
%pip install "terra-st[perturb]" gdown

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import torch
from datasets import load_from_disk

from terra import download_pretrained
from terra.inference import harmonize_adata, tokenize_adata
from terra.inference import perturb_dataset, embed_dataset, summarize_w2_by_label

sc.settings.set_figure_params(dpi=80, frameon=False)
import logging; logging.basicConfig(level="INFO")  # show TERRA progress

## 2. Configuration

Set the model, the section data file, and a local cache directory. If you host the test-section `.h5ad` on Google Drive, put its file id in `DRIVE_FILE_ID` and it will be downloaded automatically.

In [ ]:
MODEL_REPO   = "lotfollahi-lab/TERRA-96M"   # HF model bundle
SECTION      = "K014"
NICHE_KEY    = "NEMO_updated_niche"           # obs column used to group / colour
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

DATA_H5AD    = "K014_full.h5ad"          # local path to the section AnnData
DRIVE_FILE_ID = "11C6z0RZXZhbKzI8SHSRbsA_k1i1Tqsfa"  # optional: gdown fallback

CACHE        = Path("cache"); CACHE.mkdir(exist_ok=True)
TOK_CACHE    = CACHE / f"{SECTION}_tokenized"          # shared across tutorials
print("device:", DEVICE)

## 3. Download the pretrained model (Hugging Face)

`download_pretrained` fetches the model bundle (`model_checkpoint.pt`, `model_config.yaml`, `token_dictionary.pkl`) and returns its local path.

In [ ]:
model_dir = download_pretrained(MODEL_REPO)
print("model bundle:", model_dir)

## 4. Load the section data

The provided test-section `.h5ad` holds raw counts (`X`), spatial coordinates (`obsm['spatial']`) and niche / cell-type labels for the whole test section (123,563 cells).

In [ ]:
if not os.path.exists(DATA_H5AD):
    import gdown
    print("downloading section data from Google Drive ...")
    gdown.download(id=DRIVE_FILE_ID, output=DATA_H5AD, quiet=False)

adata = sc.read_h5ad(DATA_H5AD)
if "cell_id" not in adata.obs.columns:
    adata.obs["cell_id"] = adata.obs_names.astype(str)
adata.obs["cell_id"] = adata.obs["cell_id"].astype(str)
print(adata)

## 5. Harmonise & tokenise (cached)

**What:** `harmonize_adata` maps gene symbols to the model's Ensembl vocabulary and filters low-quality cells/genes; `tokenize_adata` builds the gene-token sequences the model consumes.

**Input:** raw-count AnnData.  
**Output:** the tokenised section (one row per cell).

> ⏱️ **Slow step — runs once, then cached.** Tokenising the full section (~123k cells) took **≈ 9 min** in our test run. Tokenisation is CPU-bound (it builds the spatial graph and gene-token sequences), so the time depends on your CPU cores / `nproc`, not the GPU. The result is saved to `cache/`, so re-runs — and the other tutorials — load it instantly. Delete `cache/` to recompute.

In [ ]:
adata = harmonize_adata(adata)
print(f"after harmonise: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

if TOK_CACHE.exists():
    print("loading cached tokenised dataset ...")
    tok = load_from_disk(str(TOK_CACHE))
else:
    print("tokenising (first run; will be cached) ...")
    tok = tokenize_adata(adata, model_dir, str(CACHE / "tok_tmp"), nproc=4)
    tok.save_to_disk(str(TOK_CACHE))
tok.set_format("torch", columns=["rel_x_coord", "rel_y_coord", "cell_id",
                                 "gene_tokens", "gene_expr", "n_nonzero_tokens"])
print(f"tokenised cells: {len(tok):,}")

## 6. Define & run the knockout

**What:** set the target genes to knock out (NPNT/WT1/MAGI1 glomerular filtration-barrier markers) in each cell **and** its neighbourhood, then run the perturbation.

**Input:** the tokenised section + a `perturb_df` describing the knockout.  
**Output:** `perturbed`  the *affected subset* of cells the knockout actually edits (`return_only_perturbed_cells=True`); everything downstream runs on these.

> ⏱️ **Slow step.** ≈ ** 5 minutes** on an **CPU**. Runs once; save the result to avoid recomputing.

In [ ]:
genes = ["ENSG00000168743", "ENSG00000184937", "ENSG00000151276"]  # NPNT, WT1, MAGI1
perturb_df = pd.DataFrame({
    "perturbed_cell_id": "all",
    "perturbed_ensembl_id": np.repeat(genes, 2),
    "perturbation_target": ["cell", "neighborhood"] * len(genes),
    "perturbation_type": "knockout", "foldchange": np.nan,
})
PERT_CACHE = CACHE / f"{SECTION}_perturbed"
if PERT_CACHE.exists():
    perturbed = load_from_disk(str(PERT_CACHE))
else:
    perturbed = perturb_dataset(dataset=tok, perturb_df=perturb_df, model_folder_path=model_dir,
                                nproc=1, return_only_perturbed_cells=True,
                                pad_gene_tokens=True, adjust_positions=True,
                                return_perturbation_flags=True)
    perturbed.save_to_disk(str(PERT_CACHE))
affected = list(perturbed.with_format(None)["cell_id"])
print(f"affected cells: {len(affected):,}  (of {len(tok):,})")

## 7. Build the matching control (same cells)

Subset the tokenised section to **exactly the affected cells, in the same order** as `perturbed`. This is the unperturbed baseline so baseline and perturbed are embedded on the identical cell set.

In [ ]:
pos = {c: i for i, c in enumerate(tok.with_format(None)["cell_id"])}
control = tok.select([pos[c] for c in affected])
assert list(control.with_format(None)["cell_id"]) == affected

## 8. Embed control + perturbed (affected cells only)

**What:** run the model to get an embedding for every affected cell, once for the unperturbed (control) tokens and once for the perturbed tokens.

**Input:** the aligned `control` and `perturbed` datasets.  
**Output:** `base` and `pert` embeddings (`cell_emb`, `neighborhood_emb`). Both run only over the affected subset.

> ⏱️ **Slow GPU step.** ≈ ** 17 minutes** on an **H100 80GB**. 

In [ ]:
emb_kwargs = dict(model_folder_path=model_dir, emb_layer=None, agg_excluded_genes=None,
                  top_k=None, batch_size=128, include_spatial_cell_emb=True,
                  return_token_embeddings=False, ignore_spc_tokens=True, num_workers=8)
base = embed_dataset(dataset=control,   **emb_kwargs)   # baseline, affected subset
pert = embed_dataset(dataset=perturbed, **emb_kwargs)   # perturbed, affected subset

## 9. Score the perturbation per niche (W2)

**What:** for each niche, measure how far its embeddings *moved* between the unperturbed and perturbed states, using a Wasserstein-2 (Sinkhorn) distance.

**Output:** a table of per-niche W2 scores.  
**How to read it:** a larger W2 = the knockout affected that niche more. The **glomerular** niche is expected to score highest, because we knocked out its marker genes that is the biological sanity check / ground truth.

In [ ]:
adata_sub = adata[adata.obs["cell_id"].isin(set(affected))].copy()
obs_ids = adata_sub.obs["cell_id"].astype(str)
for k in ["cell_emb", "neighborhood_emb"]:
    b = pd.DataFrame(np.asarray(base[k]), index=affected)
    p = pd.DataFrame(np.asarray(pert[k]), index=affected)
    adata_sub.obsm[k]              = b.reindex(obs_ids).to_numpy(dtype=np.float32)
    adata_sub.obsm[k + "_perturb"] = p.reindex(obs_ids).to_numpy(dtype=np.float32)

df_w2 = summarize_w2_by_label(
    adata_sub, label_key=NICHE_KEY,
    pairs=[("cell_emb", "cell_emb_perturb", "cell_emb"),
           ("neighborhood_emb", "neighborhood_emb_perturb", "neighborhood_emb")],
    agg="mean", sort_by="neighborhood_emb", ascending=False, device=DEVICE)
df_w2 = df_w2[df_w2["n_cells"] > 50]
df_w2

---

## Questions / help

If you have any questions about this tutorial, please contact **Mohammad Vali Sanian** — [mohammad.sanian@helsinki.fi](mailto:mohammad.sanian@helsinki.fi) or [mv10@sanger.ac.uk](mailto:mv10@sanger.ac.uk).